# core

> schedules, the lists they read, and the notes they leave

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

All of pobblebonk is here, because [honker](https://github.com/russellromney/honker) already has
most of it. honker brings the queue, the durable scheduler table, the streams with a cursor per
reader, the retries and the named locks, all in one SQLite file and all as SQL functions.

What is left is what honker has not got, and it is three things. A pile you add to over a week
that a fire later reads. A gate that stops a fire running when its pile is empty. And a callback,
because honker fires a job and something has to decide what a job means.

There is no daemon. `honker_scheduler_tick(now)` is a SQL function, so `tick` is one call and a
cron entry is the whole scheduler.

In [ ]:
#| export
import json, re, threading, time
from collections import Counter
from contextlib import contextmanager
from pathlib import Path
from tempfile import mkdtemp

import honker
from fastcore.all import AttrDict, L, first, ifnone, patch, dict2obj
from honker import crontab, every_s


In [ ]:
#| export
_DUR = re.compile(r'([\d.]+)\s*([smhdwMy])', re.I)
_MULT = dict(s=1, m=60, h=3600, d=86400, w=604800, M=2629800, y=31557600)

In [ ]:
#| export
def secs(every) -> int:
    "Whole seconds from `'30m'`, `'6h'`, `'1w'`, `'1h30m'`, or a number."
    try: return int(float(every))
    except (TypeError, ValueError): pass
    if not (ms := _DUR.findall(str(every))): raise ValueError(f'not a duration: {every!r}')
    return int(sum(float(n) * _MULT[u.lower()] for n, u in ms))

In [ ]:
test_eq(secs('30m'), 1800)
test_eq(secs('1h30m'), 5400)
test_eq(secs('2 days'), 172800)
test_eq(secs('2 years'), _MULT['y']*2)
test_eq(secs(90), 90)
test_fail(lambda: secs('soon'), contains='not a duration')

In [ ]:
#| export
FIRES = 'pob.fires'          # the honker queue every schedule fires into
NOTES = 'pob.notes'          # the honker stream a person reads
RETRIES = 5                  # attempts a fire gets before it is dead-lettered
BASE = 30                    # seconds before the first retry; doubles per attempt

In [ ]:
#| export
def backoff(attempt:int,       # failures so far; 1 on the first
            base:float=BASE,   # the first delay, in seconds
            cap:float=3600,    # longest a retry ever waits
            ) -> float:
    'Doubling delay: 30s, 60s, 120s, 240s, capped. A bot wall clears on its own or it does not.'
    return min(base * 2 ** max(0, attempt - 1), cap)

In [ ]:
test_eq([backoff(a) for a in (1, 2, 3, 4)], [30, 60, 120, 240])
test_eq(backoff(1, base=10), 10)
test_eq(backoff(99), 3600)                    # capped: a wall that has not cleared in an hour will not
test_eq(backoff(0), 30)                       # an attempt count of 0 is still the first delay

In [ ]:
#| export
MAX_TEXT = 20_000            # chars kept of a note body; an answer is a note, not a document
def clip(s, mx:int=MAX_TEXT) -> str:
    'Text short enough to store, saying what it dropped rather than losing it quietly.'
    s = '' if s is None else str(s)
    return s if len(s) <= mx else s[:mx] + f'\n... [{len(s)-mx} more chars]'

In [ ]:
test_eq(clip('abcdef', 3), 'abc\n... [3 more chars]')

## Pob

In [ ]:
#| export
class Pob:
    "A honker database, the one table honker has not got, and the callbacks a fire runs."
    def __init__(self,
                 path=None,          # the file; None makes a temporary one
                 db=None,            # or a honker database something else already opened
                 retries:int=RETRIES,# attempts a failing fire gets before it is dead-lettered
                 base:float=BASE,    # seconds before the first retry; doubles per attempt
                 lease:int=300):     # seconds a claim lasts; a running callback renews it
        # honker watches `PRAGMA data_version` on a file, so `:memory:` is not an option
        if db is None and path is None: path = Path(mkdtemp())/'pob.db'
        self.db = db if db is not None else honker.open(str(path))
        self.retries, self.base, self.lease = max(1, int(retries)), float(base), max(1, int(lease))
        self.q, self.stream = self.db.queue(FIRES, visibility_timeout_s=self.lease), self.db.stream(NOTES)
        self.sched, self.runners = honker.Scheduler(self.db), {}
        with self.db.transaction() as tx:
            tx.execute('CREATE TABLE IF NOT EXISTS pob_items (id INTEGER PRIMARY KEY, topic TEXT, '
                       'text TEXT, key TEXT, at REAL, used_at REAL, used_by TEXT)')
            tx.execute('CREATE INDEX IF NOT EXISTS pob_items_open ON pob_items(topic, at) '
                       'WHERE used_at IS NULL')
            tx.execute('CREATE UNIQUE INDEX IF NOT EXISTS pob_items_key ON pob_items(topic, key) '
                       'WHERE key IS NOT NULL AND used_at IS NULL')

In [ ]:
p = Pob()
test_eq(p.retries, RETRIES)
test_eq(p.base, float(BASE))
test_eq((p.lease, p.q.visibility_timeout_s), (300, 300))
test_eq(Pob(lease=2).q.visibility_timeout_s, 2)

## The notes a person reads

Nothing is sent anywhere. An agent cannot start a turn of its own, so a fire leaves a note and
each place you work drains it on its next turn. honker's stream keeps a cursor per reader, so a
phone and a terminal each see everything once.

In [ ]:
#| export
@patch
def note(self:Pob, title:str, body:str='', **meta):
    'Leave one note.'
    self.stream.publish(dict(title=str(title), body=clip(body), **meta))

def _note(r): return AttrDict(json.loads(r['payload']), offset=r['offset'])
@patch
def notes(self:Pob, limit:int=50) -> L:
    'Every note, newest first, whatever any reader has seen.'
    return L(self.db.query('SELECT offset, payload FROM _honker_stream WHERE topic=? '
                           'ORDER BY offset DESC LIMIT ?', [NOTES, limit])).map(_note)

@patch
def drain(self:Pob, reader:str, limit:int=50) -> L:
    'Every note this reader has not had. What a turn calls before it answers you.'
    got = L(self.db.query('SELECT offset, payload FROM _honker_stream WHERE topic=? AND offset>? '
                          'ORDER BY offset LIMIT ?',
                          [NOTES, self.stream.get_offset(str(reader)), limit]))
    if got: self.stream.save_offset(str(reader), got[-1]['offset'])
    return got.map(_note)

In [ ]:
p_notes = Pob()
p_notes.note('first', 'one', kind='answer')
p_notes.note('second', 'two')
print('added notes: ', p_notes.notes())
# `notes` is an inspection view: newest first
test_eq(p_notes.notes().attrgot('title'), ['second', 'first'])
test_eq(p_notes.notes()[1].kind, 'answer')
# `drain` is a reader view: unread notes, oldest first
print('phone reader drain: ', p_notes.drain('phone'))
test_eq(p_notes.drain('phone'), [])
# each reader has a separate cursor
print('terminal reader drain:', p_notes.drain('terminal'))
test_eq(p_notes.drain('terminal'), [])

added notes:  [{'title': 'second', 'body': 'two', 'offset': 2}, {'title': 'first', 'body': 'one', 'kind': 'answer', 'offset': 1}]
phone reader drain:  [{'title': 'first', 'body': 'one', 'kind': 'answer', 'offset': 1}, {'title': 'second', 'body': 'two', 'offset': 2}]
terminal reader drain: [{'title': 'first', 'body': 'one', 'kind': 'answer', 'offset': 1}, {'title': 'second', 'body': 'two', 'offset': 2}]


In [ ]:
#| export
@patch
def due(self:Pob) -> int:
    'Fires queued and not yet run.'
    return self.db.query("SELECT count(*) n FROM _honker_live WHERE queue=? AND state='pending' AND run_at <= unixepoch()",
                         [FIRES])[0]['n']
@patch
def all(self:Pob) -> L:
    'Every schedule, with its payload parsed.'
    return L(self.sched.list()).map(lambda r: AttrDict(r, payload=json.loads(r['payload'] or 'null')))

@patch
def __repr__(self:Pob):
        return f'Pob({len(self.all())} schedules, {self.due()} due, {len(self.notes())} notes)'


In [ ]:
test_eq(repr(p), 'Pob(0 schedules, 0 due, 0 notes)')

## Schedules

A schedule is a durable instruction identified by `name`. `add` creates it. `update` changes the
fields supplied by the caller and preserves the other fields. Payload changes merge with the stored
payload. The schedule name does not change.

`batch` caps how many items of `needs` one fire takes. `batch=1` runs one job per fire and leaves the rest open.

In [ ]:
#| export
CATCHUP = ('once', 'all')    # or a positive int: how many missed fires to make up
def catchup_n(v) -> int:
    "Fires kept of the ones missed while the machine was off. 0 means keep every one."
    if v == 'all': return 0
    if v == 'once': return 1
    if isinstance(v, int) and not isinstance(v, bool) and v > 0: return v
    raise ValueError(f"catchup must be 'once', 'all' or a positive int: {v!r}")

def _batch(v):
    "None, or the positive int of items a fire takes."
    if v is None: return None
    if isinstance(v, int) and not isinstance(v, bool) and v > 0: return v
    raise ValueError(f'batch must be a positive int or None: {v!r}')

@patch
def add(self:Pob,
        name:str,             # what you call it; unique, and what a note is titled with
        cron:str=None,        # a five-field cron expression
        every=None,           # or a duration: `'30m'`, `'1w'`, or seconds
        needs:str=None,       # a list topic this cannot run without
        catchup='once',       # `once` runs the latest missed fire, `all` every one, an int that many
        start:float=None,     # epoch seconds for the first fire; None lets the cadence pick it
        retries:int=None,     # attempts a failing fire gets; None -> the Pob's own
        batch:int=None,       # items of `needs` one fire takes, oldest first; None takes them all
        **payload             # anything the callback wants; `retry_after`, a duration, replaces the doubling delay before a retry
) -> AttrDict:
    'Write down a standing instruction.'
    if bool(cron) == bool(every): raise ValueError('a schedule needs one of `cron` or `every`')
    catchup_n(catchup)
    if payload.get('retry_after') is not None: secs(payload['retry_after'])
    self.sched.add(name=str(name), queue=FIRES, payload=dict(
        schedule=str(name), needs=needs, catchup=catchup, batch=_batch(batch), **payload),
        schedule=crontab(cron) if cron else every_s(secs(every)),
        max_attempts=max(1, int(ifnone(retries, self.retries))))
    if start is not None: self.fire_at(name, start)
    return self.get(name)

@patch
def fire_at(self:Pob, name:str, at:float) -> AttrDict:
    'Move the next fire of `name`. The cadence picks every boundary after it.'
    with self.db.transaction() as tx:
        tx.execute('UPDATE _honker_scheduler_tasks SET next_fire_at=? WHERE name=?',
                   [int(at), str(name)])
    return self.get(name)

@patch
def on(self:Pob, name:str):
    'Decorator naming the callback a fire of `name` runs.'
    def _f(fn):
        self.runners[str(name)] = fn
        return fn
    return _f

@patch
def get(self:Pob, name:str) -> AttrDict: return first(r for r in self.all() if r['name'] == str(name))

@patch
def pause(self:Pob, name:str) -> bool: return self.sched.pause(str(name))

@patch
def resume(self:Pob, name:str) -> bool: return self.sched.resume(str(name))

@patch
def drop(self:Pob, name:str) -> bool:
    self.runners.pop(str(name), None)
    return self.sched.remove(str(name))

@patch
def update(self:Pob,
           name:str,          # schedule to change
           cron:str=None,     # new cron expression
           every=None,        # or a new duration
           priority:int=None, # new queue priority
           retries:int=None,  # new attempt budget
           **payload          # payload fields to add or replace, `batch` among them
) -> AttrDict:
    'Change an existing schedule without replacing fields that are not given.'
    if cron is not None and every is not None: raise ValueError('a schedule update accepts only one of `cron` or `every`')
    if 'catchup' in payload: catchup_n(payload['catchup'])
    if 'batch' in payload: _batch(payload['batch'])
    if payload.get('retry_after') is not None: secs(payload['retry_after'])
    if (was := self.get(name)) is None: return None
    changes = {}
    if cron is not None: changes['schedule'] = crontab(cron)
    elif every is not None: changes['schedule'] = every_s(secs(every))
    if priority is not None: changes['priority'] = int(priority)
    if retries is not None: changes['max_attempts'] = max(1, int(retries))
    if payload: changes['payload'] = {**(was.payload or {}), **payload, 'schedule': str(name)}
    self.sched.update(str(name), **changes)
    return self.get(name)


In [ ]:
p_sched = Pob()

added = p_sched.add('digest', every='1h', needs='inbox', catchup='once', retries=4, tone='brief')

test_eq(p_sched.get('digest').name, 'digest')
test_eq(len(p_sched.all()), 1)
test_eq(added.payload, {'schedule': 'digest','needs': 'inbox','catchup': 'once','batch': None,'tone': 'brief'})

changed = p_sched.update('digest',every='2h',priority=9,retries=7,catchup='all',tone='detailed')

test_eq(changed.cron_expr, every_s(secs('2h')).expr)
test_eq(changed.priority, 9)
test_eq(changed.max_attempts, 7)
test_eq(changed.payload, {'schedule': 'digest','needs': 'inbox','catchup': 'all','batch': None,'tone': 'detailed'})
test_eq(p_sched.update('digest', batch=1).payload['batch'], 1)
test_eq(p_sched.update('digest', batch=None).payload['batch'], None)
test_fail(lambda: p_sched.update('digest', batch=0), contains='batch must be')
test_fail(lambda: p_sched.add('bad', every='1h', batch='1'), contains='batch must be')
test_eq(p_sched.update('digest', retries=2, retry_after='1h').payload['retry_after'], '1h')
test_fail(lambda: p_sched.update('digest', retry_after='soon'), contains='not a duration')

## The lists a fire reads

Reading a list never empties it. Only the work that finished does that, so a cart run that failed
loses no milk.

In [ ]:
#| export
@patch
def push(self:Pob,
         topic:str,      # which list
         text:str,       # the line the callback reads
         key:str=None,   # send the same thing twice and it stays one item
         at:float=None
) -> AttrDict:
    'Add one item, or give back the open one that `key` already names.'
    at = ifnone(at, time.time())
    with self.db.transaction() as tx:
        if key is not None and (was := tx.query(
                'SELECT * FROM pob_items WHERE topic=? AND key=? AND used_at IS NULL',
                [str(topic), key])): return AttrDict(was[0])
        got = tx.query('INSERT INTO pob_items (topic, text, key, at) VALUES (?,?,?,?) RETURNING *',
                       [str(topic), clip(text), key, at])
    return AttrDict(got[0])

@patch
def items(self:Pob, topic:str, limit:int=200) -> L:
    'The open items on `topic`, oldest first, in push order on a tie.'
    return L(self.db.query('SELECT * FROM pob_items WHERE topic=? AND used_at IS NULL '
                           'ORDER BY at, id LIMIT ?', [str(topic), limit])).map(AttrDict)

In [ ]:
p_items = Pob()

milk = p_items.push('shopping', 'milk', at=20)
bread = p_items.push('shopping', 'bread', key='photo-1', at=10)
same_bread = p_items.push('shopping', 'bread again', key='photo-1', at=30)
p_items.push('reading', 'Dune', at=5)

```python
{ 'at': 5.0,
  'id': 3,
  'key': None,
  'text': 'Dune',
  'topic': 'reading',
  'used_at': None,
  'used_by': None}
```

In [ ]:
# `key` makes repeated open items idempotent
test_eq(same_bread.id, bread.id)
test_eq(same_bread.text, 'bread')

# items are separate by topic and ordered oldest first
test_eq(p_items.items('shopping').attrgot('text'), ['bread', 'milk'])
test_eq(p_items.items('reading').attrgot('text'),['Dune'])

# limit applies after ordering
test_eq(p_items.items('shopping', limit=1).attrgot('text'),['bread'])

`used` records that an item was consumed successfully. A used item no longer appears in `items`.
`used_by` is free-form provenance supplied by the caller. Pobblebonk stores it for inspection. It
does not validate the value or use it for access control. `drop_item` deletes an open item; a used one stays.

In [ ]:
#| export
@patch
def used(self:Pob, ids, by:str='', at:float=None) -> int:
    'Mark items done, once the work that consumed them succeeded.'
    if not (ids := L(ids)): return 0
    qs = ','.join('?'*len(ids))
    with self.db.transaction() as tx:
        return len(tx.query(f'UPDATE pob_items SET used_at=?, used_by=? WHERE id IN ({qs}) '
                            'AND used_at IS NULL RETURNING id',
                            [ifnone(at, time.time()), str(by), *ids]))

@patch
def drop_item(self:Pob, id:int) -> bool:
    'Delete an open item, a job cancelled before it starts.'
    with self.db.transaction() as tx:
        return bool(tx.query('DELETE FROM pob_items WHERE id=? AND used_at IS NULL RETURNING id', [int(id)]))

In [ ]:
# no IDs means no change
test_eq(p_items.used([]), 0)
# mark only bread as used
test_eq(p_items.used([bread.id], by='cart', at=100), 1)
test_eq(p_items.items('shopping').attrgot('text'), ['milk'])

used_bread = AttrDict(p_items.db.query('SELECT used_at, used_by FROM pob_items WHERE id=?',[bread.id])[0])
test_eq((used_bread.used_at, used_bread.used_by), (100.0, 'cart'))

# marking the same item twice changes nothing
test_eq(p_items.used([bread.id], by='other', at=200), 0)

# its old key is now free for a new open item
new_bread = p_items.push('shopping', 'fresh bread', key='photo-1', at=30)
test_ne(new_bread.id, bread.id)
test_eq(p_items.items('shopping').attrgot('text'),['milk', 'fresh bread'])

# `drop_item` deletes an open item and leaves a used one alone
test_eq(p_items.drop_item(new_bread.id), True)
test_eq(p_items.drop_item(new_bread.id), False)
test_eq(p_items.drop_item(bread.id), False)
test_eq(p_items.items('shopping').attrgot('text'),['milk'])

## The tick

One SQL call fires what is due and advances every `next_fire_at`. honker enqueues a job for every
missed boundary, so `once` keeps the latest of them and cancels the rest.

In [ ]:
#| export
def superseded(fires, keep) -> set:
    'Older job IDs for schedules that make up only their most recent missed fires.'
    fires = L(fires)
    keep = keep if isinstance(keep, dict) else {n: 1 for n in keep}
    return {
        f['job_id']
        for name, n in keep.items()
        for f in fires.filter(lambda f: f['name'] == name)[:-int(n)]
    }


In [ ]:
fires = L([
    {'name': 'digest', 'job_id': 1, 'fire_at': 10},
    {'name': 'digest', 'job_id': 2, 'fire_at': 20},
    {'name': 'digest', 'job_id': 3, 'fire_at': 30},
    {'name': 'archive', 'job_id': 4, 'fire_at': 10},
    {'name': 'archive', 'job_id': 5, 'fire_at': 20},
])

# `digest` runs once, so only its newest fire remains
test_eq(superseded(fires, {'digest'}), {1, 2})
# `archive` uses catchup='all', so none of its fires are superseded
test_eq(superseded(fires, set()), set())
# one fire has nothing older to replace
test_eq(superseded([{'name': 'digest', 'job_id': 1, 'fire_at': 10}], {'digest'}), set())
# a count keeps that many of the missed fires, newest last
test_eq(superseded(fires, {'digest': 2}), {1})
test_eq(superseded(fires, {'digest': 3}), set())
test_eq(superseded(fires, {'digest': 1, 'archive': 1}), {1, 2, 4})


In [ ]:
#| export
@patch
def tick(
    self:Pob,
    at:float=None,        # when this tick is happening; now when unset
    worker:str='tick',    # the lease holder recorded on each job
    run:bool=True,        # False queues the fires and runs none of them
) -> AttrDict:
    'Create due fires, apply catch-up policy, then run the remaining jobs.'
    at = int(ifnone(at, time.time()))
    with self.db.transaction() as tx: fires = L(json.loads(tx.query('SELECT honker_scheduler_tick(?) AS f',[at])[0]['f']))
    keep = {s.name: n for s in self.all() if (n := catchup_n(s.payload.get('catchup', 'once')))}
    dropped = {job_id for job_id in superseded(fires, keep) if self.q.cancel(job_id)}
    kept = fires.filter(lambda fire: fire['job_id'] not in dropped)
    missed = dict(Counter(f['name'] for f in fires if f['job_id'] in dropped))
    fire_times = {fire['job_id']: fire['fire_at'] for fire in kept}
    return dict2obj(dict(fired=kept,cancelled=len(dropped),missed=missed,ran=self.work(worker, at, when=fire_times) if run else L()))


In [ ]:
# A tick before the first boundary creates no jobs.
p_early = Pob()
p_early.add('early', every='1s')
first_fire = p_early.get('early').next_fire_at
first_fire

1787713032

In [ ]:
early = p_early.tick(at=first_fire - 1, run=False)
test_eq(early.fired, [])
test_eq(early.cancelled, 0)
test_eq(early.ran, [])

In [ ]:
# `once` keeps only the newest missed boundary.
p_once = Pob()
p_once.add('digest', every='1s', catchup='once')
first_fire = p_once.get('digest').next_fire_at
once = p_once.tick(at=first_fire + 2, run=False)
once

```python
{ 'cancelled': 2,
  'fired': [{'fire_at': 1787713034, 'job_id': 3, 'name': 'digest', 'queue': 'pob.fires'}],
  'ran': []}
```

In [ ]:
test_eq(len(once.fired), 1)
test_eq(once.cancelled, 2)
test_eq(once.fired[0].fire_at, first_fire + 2)
test_eq(once.ran, [])

In [ ]:
# `all` keeps every missed boundary.
p_all = Pob()
p_all.add('archive', every='1s', catchup='all')
first_fire = p_all.get('archive').next_fire_at
first_fire

1787713032

In [ ]:
all_fires = p_all.tick(at=first_fire + 2, run=False); all_fires

```python
{ 'cancelled': 0,
  'fired': [{'fire_at': 1787713032, 'job_id': 1, 'name': 'archive', 'queue': 'pob.fires'}, {'fire_at': 1787713033, 'job_id': 2, 'name': 'archive', 'queue': 'pob.fires'}, {'fire_at': 1787713034, 'job_id': 3, 'name': 'archive', 'queue': 'pob.fires'}],
  'ran': []}
```

In [ ]:
test_eq(len(all_fires.fired), 3)
test_eq(all_fires.cancelled, 0)
test_eq(all_fires.fired.attrgot('fire_at'),[first_fire, first_fire + 1, first_fire + 2])
test_eq(all_fires.ran, [])

A count between the two: make up that many missed boundaries and drop the rest.


In [ ]:
p_two = Pob()
p_two.add('digest', every='1s', catchup=2)
_f = p_two.get('digest').next_fire_at
two = p_two.tick(at=_f + 2, run=False)
test_eq(len(two.fired), 2)
test_eq(two.cancelled, 1)
test_eq(two.missed, {'digest': 1})   # which schedule lost a boundary, not just how many
test_eq(two.fired.attrgot('fire_at'), [_f + 1, _f + 2])   # the most recent two
test_fail(lambda: p_two.add('bad', every='1s', catchup=0), contains='positive int')
test_fail(lambda: p_two.add('bad', every='1s', catchup='some'), contains='positive int')


`start` sets the first fire. Without it the cadence picks the next boundary from now.


In [ ]:
p_start = Pob()
_at = int(time.time()) + 3600
test_eq(p_start.add('later', every='1h', start=_at).next_fire_at, _at)
test_eq(p_start.tick(at=_at - 1, run=False).fired, [])          # nothing is due yet
test_eq(len(p_start.tick(at=_at, run=False).fired), 1)          # and then it is
test_eq(p_start.fire_at('later', _at + 60).next_fire_at, _at + 60)


A claimed Honker `Job` does not expose the stored values for `max_attempts`, `created_at`, or
`run_at`. `_row` reads the live queue row when callback or retry logic needs those values.

In [ ]:
#| export
@patch
def _row(self:Pob, job) -> AttrDict:
    'fetch specific honker job'
    return AttrDict(first(self.db.query('SELECT * FROM _honker_live WHERE id=?', [job.id])) or {})

In [ ]:
p_row = Pob(retries=7)
p_row.q.enqueue({'schedule': 'x'}, max_attempts=7, run_at=1_700_000_000)
job = p_row.q.claim_one('w')
test_eq((job.max_attempts, job.created_at, job.run_at), (3, 0, 0))
row = p_row._row(job)
test_eq(row.max_attempts, 7)
test_eq(row.run_at, 1_700_000_000)
test_eq(row.created_at > 0, True)

A callback failure spends one attempt. While the stored budget has attempts left, `_failed`
returns the job to the queue after a doubling delay, or after the schedule's own `retry_after`
when it has one: a fire that failed for want of something a person fixes is worth one try an
hour later, not four in the next eight minutes. The final failure dead-letters the job and
leaves a note, because it will never produce an ordinary answer.

In [ ]:
#| export
@patch
def _failed(self:Pob, job, row, name:str, exc:Exception) -> AttrDict:
    'Retry a failed callback while attempts remain, else dead-letter it.'
    err = f'{type(exc).__name__}: {exc}'
    left = max(0, int(row.get('max_attempts') or job.max_attempts or 1) - job.attempts)
    if left:
        delay = secs(ra) if (ra := (job.payload or {}).get('retry_after')) else backoff(job.attempts, self.base)
        if job.retry(int(delay),err): return AttrDict(name=name,status='retry',error=err,attempt=job.attempts,attempts_left=left,after=delay)
    job.fail(err)
    self.note(name, f'gave up after {job.attempts} tries. {err}', ref=job.id, dead=True)
    return AttrDict(name=name, status='error', error=err, attempt=job.attempts)

In [ ]:
p_retry = Pob(base=10)
p_retry.q.enqueue({'schedule': 'flaky'}, max_attempts=5)
job = p_retry.q.claim_one('worker')
row = p_retry._row(job)
retry = p_retry._failed(job, row, 'flaky', RuntimeError('bot wall')); retry

```python
{ 'after': 10.0,
  'attempt': 1,
  'attempts_left': 4,
  'error': 'RuntimeError: bot wall',
  'name': 'flaky',
  'status': 'retry'}
```

In [ ]:
test_eq((retry.status, retry.error, retry.attempt, retry.attempts_left, retry.after), ('retry', 'RuntimeError: bot wall', 1, 4, 10.0))
test_eq(p_retry.notes(), [])
test_eq(p_retry.db.query('SELECT count(*) n FROM _honker_dead')[0]['n'], 0)

In [ ]:
p_later = Pob(base=10)
p_later.q.enqueue({'schedule': 'nightly', 'retry_after': '1h'}, max_attempts=2)
job = p_later.q.claim_one('worker')
test_eq(p_later._failed(job, p_later._row(job), 'nightly', RuntimeError('no claude')).after, 3600)

In [ ]:
p_dead = Pob(base=10)
p_dead.q.enqueue({'schedule': 'flaky'}, max_attempts=1)
job = p_dead.q.claim_one('worker')
row = p_dead._row(job)
failed = p_dead._failed(job, row, 'flaky', ValueError('bad answer')); failed

```python
{ 'attempt': 1,
  'error': 'ValueError: bad answer',
  'name': 'flaky',
  'status': 'error'}
```

In [ ]:
test_eq((failed.status, failed.error, failed.attempt), ('error', 'ValueError: bad answer', 1))
test_eq(p_dead.db.query('SELECT count(*) n FROM _honker_dead')[0]['n'], 1)

note = p_dead.notes()[0]
test_eq((note.title, note.dead, note.ref), ('flaky', True, job.id))
test_eq(note.body, 'gave up after 1 tries. ValueError: bad answer')

A successful callback has four effects: its result is stored, consumed list items are closed,
a note is published, and the job is acknowledged. `_done` keeps those effects together. A callback that returns `None` or `''` and used no items has nothing to say, so it leaves no note.

In [ ]:
#| export
@patch
def _done(self:Pob, job, name:str, got, res) -> AttrDict:
    'Store a successful callback result and finish its fire.'
    self.q.save_result(job.id, res)
    if got: self.used(got.attrgot('id'), by=name)
    if res not in (None, '') or got:
        self.note(name, res if isinstance(res, str) else json.dumps(res, default=str), ref=job.id, used=len(got))
    job.ack()
    return AttrDict(name=name, status='ok', result=res, used=len(got))

In [ ]:
p_done = Pob()
milk = p_done.push('shopping', 'milk')
job_id = p_done.q.enqueue({'schedule': 'cart'})
job = p_done.q.claim_one('worker')
got = p_done.items('shopping')

done = p_done._done(job, 'cart', got, {'added': 1})

In [ ]:
test_eq(done, {'name': 'cart','status': 'ok','result': {'added': 1},'used': 1})
test_eq(p_done.items('shopping'), [])
note = p_done.notes()[0]
test_eq((note.title, note.body, note.ref, note.used),('cart', '{"added": 1}', job.id, 1))

A fire that returns `None` and uses no items leaves no note: not the text `null`.

In [ ]:
p_quiet = Pob()
p_quiet.q.enqueue({'schedule': 'index'})
job = p_quiet.q.claim_one('worker')
test_eq(p_quiet._done(job, 'index', L(), None).status, 'ok')
test_eq(p_quiet.notes(), [])

Honker stores the schedule fields in a job payload, while the callback also needs the actual fire
time, its open list items, and the job ID. `_fire` gathers those pieces into the object the callback
receives. `batch` caps the list. `fire.worker` names the claim holder. `fire.beat()` renews the claim and
`fire.note(body)` leaves a progress note with the job ID as its `ref`.

In [ ]:
#| export
@patch
def _fire(self:Pob, job, at:int, fire_at:int=None, worker:str=None):
    'Build the value passed to a fire callback.'
    p, row, worker = AttrDict(job.payload or {}), self._row(job), ifnone(worker, job.worker_id)
    got = self.items(p.needs, limit=p.get('batch') or 200) if p.get('needs') else L()
    fire_at = ifnone(fire_at, row.get('run_at') or at)
    beat = lambda: self.q.heartbeat(job.id, worker, extend_s=self.lease)
    note = lambda body, **meta: self.note(p.get('schedule') or '', body, ref=job.id, **meta)
    return AttrDict(p, at=at, fire_at=fire_at, list=got, job=job.id, worker=worker, beat=beat, note=note), row

In [ ]:
p_fire = Pob()
item = p_fire.push('shopping', 'milk')
job_id = p_fire.q.enqueue({'schedule': 'cart', 'needs': 'shopping'}, run_at=100)
job = p_fire.q.claim_one('worker')

fire, row = p_fire._fire(job, at=200)

In [ ]:
test_eq((fire.schedule, fire.needs), ('cart', 'shopping'))
test_eq(fire.list.attrgot('id'), [item.id])
test_eq((fire.job, fire.fire_at), (job.id, 100))
test_eq(row.id, job.id)
test_eq(fire.worker, 'worker')
test_eq(fire.beat(), True)
fire.note('half done', step=1)
n = p_fire.notes()[0]
test_eq((n.title, n.body, n.ref, n.step), ('cart', 'half done', job.id, 1))

# An explicit scheduler boundary takes precedence over the stored run time.
fire, _ = p_fire._fire(job, at=200, fire_at=50)
test_eq(fire.fire_at, 50)

While a callback runs, `_beating` renews its claim every third of the lease, so a long job keeps it.
The thread stops when the callback returns or raises.

In [ ]:
#| export
@contextmanager
def _beating(q, job_id:int, worker:str, lease:int):
    'Renew a claim every `lease/3` seconds until the block exits.'
    stop = threading.Event()
    def _beat():
        while not stop.wait(lease/3): q.heartbeat(job_id, worker, extend_s=lease)
    t = threading.Thread(target=_beat, name=f'pob.beat.{job_id}', daemon=True)
    t.start()
    try: yield t
    finally: stop.set(); t.join()

In [ ]:
#| export
@patch
def _run(self:Pob, job, at:int, fire_at:int=None, worker:str=None) -> AttrDict:
    'Run one claimed fire, renewing its claim while the callback runs.'
    p, worker = AttrDict(job.payload or {}), ifnone(worker, job.worker_id)
    name = p.get('schedule') or ''
    if (fn := self.runners.get(name)) is None:
        job.fail(err := f'no callback registered for {name!r}')
        self.note(name, err, ref=job.id, dead=True)
        return AttrDict(name=name, status='error', error=err)
    fire, row = self._fire(job, at, fire_at, worker)
    got = fire.list
    if fire.get('needs') and not got:
        job.ack()
        return AttrDict(name=name, status='skipped', why=f'the {fire.needs} list is empty')
    try:
        with _beating(self.q, job.id, worker, self.lease): res = fn(fire)
    except Exception as e: return self._failed(job, row, name, e)
    return self._done(job, name, got, res)

In [ ]:
p_missing = Pob()
p_missing.q.enqueue({'schedule': 'nobody'})
job = p_missing.q.claim_one('worker')

missing = p_missing._run(job, at=100)

test_eq(missing.status, 'error')
test_eq(missing.error, "no callback registered for 'nobody'")
test_eq(p_missing.notes()[0].body, missing.error)
test_eq(p_missing.notes()[0].dead, True)

In [ ]:
p_empty = Pob()
called = []
p_empty.on('cart')(lambda fire: called.append(fire) or 'done')
p_empty.q.enqueue({'schedule': 'cart', 'needs': 'shopping'})
job = p_empty.q.claim_one('worker')

empty = p_empty._run(job, at=100)

test_eq(empty.status, 'skipped')
test_eq(empty.why, 'the shopping list is empty')
test_eq(called, [])
test_eq(p_empty.notes(), [])

In [ ]:
p_raise = Pob(base=10)
p_raise.on('cart')(lambda fire: 1/0)
p_raise.push('shopping', 'milk')
p_raise.q.enqueue({'schedule': 'cart', 'needs': 'shopping'}, max_attempts=3)
job = p_raise.q.claim_one('worker')
raised = p_raise._run(job, at=100); raised

```python
{ 'after': 10.0,
  'attempt': 1,
  'attempts_left': 2,
  'error': 'ZeroDivisionError: division by zero',
  'name': 'cart',
  'status': 'retry'}
```

In [ ]:
test_eq((raised.status, raised.attempts_left, raised.after), ('retry', 2, 10.0))
test_eq(raised.error, 'ZeroDivisionError: division by zero')
test_eq(p_raise.items('shopping').attrgot('text'), ['milk'])

`work` drains ready fires from the queue. Each claim gives one worker exclusive ownership of a
job; `_run` then dispatches it to the callback named in its payload. `limit` bounds one drain, and
`when` preserves the scheduler boundary supplied by `tick`.

In [ ]:
#| export
@patch
def work(self:Pob, worker:str='tick', at:float=None, limit:int=100, when:dict=None) -> L:
    'Claim and run every fire that is due.'
    at, when, out = int(ifnone(at, time.time())), when or {}, L()
    while len(out) < limit:
        if (job := self.q.claim_one(worker)) is None: break
        out.append(self._run(job, at, when.get(job.id), worker))
    return out

In [ ]:
p_work = Pob()
seen = []

@p_work.on('task')
def task(fire):
    seen.append((fire.job, fire.fire_at))
    return fire.job

job1 = p_work.q.enqueue({'schedule': 'task'})
job2 = p_work.q.enqueue({'schedule': 'task'})
when = {job1: 10, job2: 20}

In [ ]:
# `limit` leaves the second ready job for another drain.
f = p_work.work('worker', at=100, limit=1, when=when); f

[{'name': 'task', 'status': 'ok', 'result': 1, 'used': 0}]

In [ ]:
test_eq(f.attrgot('result'), [job1])
test_eq(seen, [(job1, 10)])

In [ ]:
# The next drain claims what remains.
second = p_work.work('worker', at=100, when=when)
test_eq(second.attrgot('result'), [job2])
test_eq(seen, [(job1, 10), (job2, 20)])

# An empty queue returns immediately.
test_eq(p_work.work('worker', at=100), [])

## Long jobs

A claim lasts `lease` seconds. While a callback runs, its claim is renewed, so a job that takes an hour
is not claimed again by the next `work`. A worker that dies stops renewing, and after the lease the fire
is claimable again. Here the lease is two seconds and the job takes three.

In [ ]:
def beating(): return [t for t in threading.enumerate() if t.name.startswith('pob.beat.')]

p_long = Pob(lease=2)
@p_long.on('render')
def render(fire):
    time.sleep(3)
    return 'rendered'

p_long.q.enqueue({'schedule': 'render'})
ran = []
t = threading.Thread(target=lambda: ran.extend(p_long.work('studio')))
t.start()
time.sleep(0.5)
test_eq(len(beating()), 1)                    # the claim is being renewed
time.sleep(2.5)                               # past the lease
test_eq(p_long.work('other'), [])             # and nobody else gets the job
t.join()
test_eq([r.status for r in ran], ['ok'])
test_eq(beating(), [])                        # the renewing stopped on return
test_eq(p_long.db.query('SELECT count(*) n FROM _honker_live')[0]['n'], 0)

The renewing stops when the callback raises, too:

In [ ]:
p_long.on('broken')(lambda fire: 1/0)
p_long.q.enqueue({'schedule': 'broken'})
test_eq(p_long.work('studio')[0].status, 'retry')
test_eq(beating(), [])

A worker killed mid-job leaves its claim to lapse, and the fire runs again:

In [ ]:
p_crash = Pob(lease=2)
p_crash.on('render')(lambda fire: f'rendered by {fire.worker}')
p_crash.q.enqueue({'schedule': 'render'})
test_ne(p_crash.q.claim_one('dead'), None)    # claimed, then the process died
test_eq(p_crash.work('alive'), [])            # the claim still holds
time.sleep(3.1)
test_eq(p_crash.work('alive').attrgot('result'), ['rendered by alive'])

`batch` takes part of a list. With `batch=1` each fire gets one item, in push order, and marks only that
one used. A raise leaves its item open for the retry.

In [ ]:
p_batch = Pob(base=60)
p_batch.on('job')(lambda fire: 1/0)
p_batch.add('job', every='1s', needs='jobs', batch=1)
for t in 'abc': p_batch.push('jobs', t, key=t)
test_eq(p_batch.tick(at=p_batch.get('job').next_fire_at).ran.attrgot('status'), ['retry'])
test_eq(p_batch.items('jobs').attrgot('text'), ['a', 'b', 'c'])   # the raise left its item open
test_eq(p_batch.push('jobs', 'a again', key='a').text, 'a')       # and `key` still holds
test_eq(len(p_batch.items('jobs')), 3)

In [ ]:
p_three = Pob()
p_three.on('job')(lambda fire: ','.join(fire.list.attrgot('text')))
p_three.add('job', every='1s', needs='jobs', catchup='all', batch=1)
for t in 'abc': p_three.push('jobs', t)
_f = p_three.get('job').next_fire_at
got = p_three.tick(at=_f + 2).ran
test_eq(got.attrgot('result'), ['a', 'b', 'c'])           # three fires, one item each, in order
test_eq(got.attrgot('used'), [1, 1, 1])

## A callback, at a cadence

The headline. A real callback, a real one-second schedule, three real ticks. `fire_at` is honker's
own boundary, so the gaps say whether the cadence held.

In [ ]:
import time

pob = Pob()
beats = []

@pob.on('heartbeat')
def beat(fire):
    beats.append(fire.fire_at)
    return f'beat {len(beats)}'

pob.add('heartbeat', every='1s')
first_fire = pob.get('heartbeat').next_fire_at
for at in range(first_fire, first_fire + 3): test_eq(pob.tick(at=at).ran.attrgot('status'), ['ok'])
test_eq(beats, [first_fire, first_fire + 1, first_fire + 2])
test_eq(pob.notes().attrgot('body'), ['beat 3', 'beat 2', 'beat 1'])

The answers reached the notes, and every reader gets them once:

In [ ]:
test_eq(pob.drain('leela').attrgot('body'), ['beat 1', 'beat 2', 'beat 3'])
test_eq(pob.drain('leela'), [])                                    # leela is up to date
test_eq(len(pob.drain('phone')), 3)                                # the phone has seen nothing yet

## A list a fire reads

The shopping flow. Push whenever you think of something, and the fire reads the pile.

In [ ]:
pob2 = Pob()

@pob2.on('cart')
def cart(fire):
    return 'added: ' + ', '.join(i.text for i in fire.list)

pob2.add('cart', every='1s', needs='shopping')
pob2.push('shopping', 'yoghurt, the greek one', key='img_2201.heic')
pob2.push('shopping', 'yoghurt, the greek one', key='img_2201.heic')   # the same photo twice
pob2.push('shopping', 'oat milk')
test_eq(pob2.items('shopping').attrgot('text'), ['yoghurt, the greek one', 'oat milk'])

`fire.list` is the pile, not `dict.items`. That collision cost a real bug, so it is asserted:

In [ ]:
time.sleep(1.05)
got = pob2.tick().ran[0]
test_eq(got.status, 'ok')
test_eq(got.result, 'added: yoghurt, the greek one, oat milk')
test_eq(got.used, 2)
test_eq(pob2.items('shopping'), [])              # used, but only because the callback returned

A fire whose list emptied in the meantime is skipped, not failed. There is nothing wrong and
nothing to say:

In [ ]:
time.sleep(1.05)
got = pob2.tick().ran[0]
test_eq(got.status, 'skipped')
test_eq(got.why, 'the shopping list is empty')
test_eq(len(pob2.notes()), 1)                    # and no second note

A schedule with no callback registered fails loudly rather than vanishing:

In [ ]:
pob4 = Pob()
pob4.add('nobody', every='1s')
time.sleep(1.05)
test_eq(pob4.tick().ran[0].error, "no callback registered for 'nobody'")
test_eq(pob4.notes()[0].body, "no callback registered for 'nobody'")   # and it leaves a note

## Pausing, dropping, and what is refused

In [ ]:
pob5 = Pob()
pob5.on('cart')(lambda fire: 'ok')
pob5.add('cart', every='1s')
test_eq(pob5.pause('cart'), True)
time.sleep(1.05)
test_eq(pob5.tick().ran, [])
test_eq(pob5.resume('cart'), True)
test_eq(pob5.drop('cart'), True)
test_eq(pob5.all(), [])

test_fail(lambda: pob5.add('bad', every='1h', cron='0 17   *'), contains='one of')
test_fail(lambda: pob5.add('bad'), contains='one of')
test_fail(lambda: pob5.add('bad', every='1h', catchup='maybe'), contains='catchup must be')
test_fail(lambda: pob5.add('bad', cron='every friday'), contains='cron')

## Sharing the file

Every method takes a honker database rather than opening one, so pobblebonk can live in a file
something else already has open. The fire and the write that caused it commit together.

In [ ]:
import honker

db = honker.open(str(Path(mkdtemp())/'app.db'))
with db.transaction() as tx:
    tx.execute('CREATE TABLE docs (id TEXT PRIMARY KEY, text TEXT)')

pob6 = Pob(db=db)
pob6.on('index')(lambda fire: 'indexed')
pob6.add('index', every='1s')
test_eq(pob6.db is db, True)
test_eq(len(pob6.all()), 1)

In [ ]:
test_fail(lambda: pob5.add('bad', every='1h', cron='0 17 * * *'), contains='one of')
test_fail(lambda: pob5.add('bad'), contains='one of')
test_fail(lambda: pob5.add('bad', every='1h', catchup='maybe'), contains='catchup must be')
test_fail(lambda: pob5.add('bad', cron='every friday'), contains='cron')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()